# XGBoost invoice experiment

## Goal

Train the XGBoost baseline on the first 1,024 synthetic invoices. Predict `paid_late` and `is_fraud` for the same 128 held-out invoices used by RPT. This notebook writes `data/synthetic_invoices/xgboost_results.csv` for the comparison notebook.

The data and fraud labels are synthetic. The fixed split is an experiment demonstration, not evidence of production performance.

## 1. Load the shared split

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

ROOT = next(path for path in (Path.cwd(), Path.cwd().parent) if (path / "data/synthetic_invoices/payment_behavior.csv").exists())
sys.path.insert(0, str(ROOT / "scripts"))
from generate_synthetic_invoices import CONTEXT_ROWS, PREDICTION_ROWS, TARGETS
from run_xgboost_invoices import DATA, load_split, train_models, predict_frame

train, test = load_split()
assert len(train) == CONTEXT_ROWS and len(test) == PREDICTION_ROWS
assert train.invoice_id.is_unique and test.invoice_id.is_unique
print(f"Train: {len(train)} invoices | held-out test: {len(test)} invoices")

Train: 1024 invoices | held-out test: 128 invoices


## 2. Train, predict, and save

The runner excludes `invoice_id` and both targets from its feature matrix. It trains one classifier per target.

In [2]:
models, feature_columns = train_models(train)
results = predict_frame(models, feature_columns, test)
assert results.invoice_id.tolist() == test.invoice_id.tolist()
output_path = DATA / "xgboost_results.csv"
results.to_csv(output_path, index=False)
print(f"Saved {len(results)} predictions to {output_path.relative_to(ROOT)}")

Saved 128 predictions to data/synthetic_invoices/xgboost_results.csv


## 3. Review held-out results

In [3]:
metric_rows = []
for target in TARGETS:
    actual = results[f"actual_{target}"]
    predicted = results[f"predicted_{target}"]
    metric_rows.append({
        "Target": target,
        "Positive cases": int(actual.sum()),
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "ROC AUC": roc_auc_score(actual, results[f"probability_{target}"]),
    })
display(pd.DataFrame(metric_rows).set_index("Target").round(3))
display(results[["invoice_id", "actual_paid_late", "predicted_paid_late", "actual_is_fraud", "predicted_is_fraud"]].head(8))

,Positive cases,Accuracy,Precision,Recall,ROC AUC
Target,,,,,
paid_late,51,0.656,0.561,0.627,0.724
is_fraud,8,0.906,0.250,0.250,0.699


,invoice_id,actual_paid_late,predicted_paid_late,actual_is_fraud,predicted_is_fraud
1024,SYN-01025,1,0,0,0
1025,SYN-01026,0,0,1,0
1026,SYN-01027,0,0,0,0
1027,SYN-01028,0,0,0,0
1028,SYN-01029,1,0,0,0
1029,SYN-01030,0,0,0,0
1030,SYN-01031,0,0,0,0
1031,SYN-01032,0,0,0,0


## Next step

Run `rpt_btp_run.ipynb` after configuring your BTP deployment. Then open `compare_saved_results.ipynb`; it reads saved files and does not make an inference call.